# 06 Interpretability Analysis

This notebook uses per-group full-model evaluation results to analyze where the recommender performs better or worse: time of day, age group, and movie genre.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src import config
from src.analysis import analyze_performance_by_time_of_day, analyze_performance_by_age_group, analyze_performance_by_genre
from src.utils import ensure_dir

ensure_dir(config.FIGURES_DIR)

## Load Full Model Per-Group Results

Run notebook 05 first so `results/full_model_test_per_group.csv` exists.

In [ ]:
per_group = pd.read_csv(config.RESULTS_DIR / 'full_model_test_per_group.csv')
ablation = pd.read_csv(config.RESULTS_DIR / 'ablation_results.csv')
display(per_group.head())
display(ablation[['model', 'NDCG@10', 'delta_NDCG@10_vs_baseline', 'interpretation']])

## Performance by Time of Day

This shows whether the model ranks the true item more effectively during morning, afternoon, evening, or night contexts.

In [ ]:
time_summary = analyze_performance_by_time_of_day(per_group)
display(time_summary.sort_values('ndcg_at_10', ascending=False))

## Performance by Age Group

This checks whether demographic groups differ in ranking quality. Differences can reflect data density, preference diversity, or coarse metadata effects.

In [ ]:
age_summary = analyze_performance_by_age_group(per_group)
display(age_summary.sort_values('ndcg_at_10', ascending=False))

## Performance by Genre

The primary genre is the first active genre column on the true movie. This is a simple interpretation proxy for which item categories are easier or harder to recommend.

In [ ]:
genre_summary = analyze_performance_by_genre(per_group)
display(genre_summary)

## Context Contribution

Use the ablation table to identify which context group matters most. If a single-context model improves NDCG@10 over baseline, that context contains useful ranking signal. If the full model improves further, the context groups are complementary. If not, some metadata may be noisy, too coarse, or redundant with user/movie embeddings.

Limitations: MovieLens 100K is small by modern standards, does not include real device/session context, and negative sampling evaluates sampled ranking rather than ranking against every unseen movie.